# Inicios

In [24]:
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import opensim as osim
import time
import os
from tqdm import tqdm
import scipy
from scipy import signal
from scipy.signal import find_peaks
import pickle
from scipy.signal.windows import dpss
from scipy.fft import rfft, rfftfreq
osim.GetVersionAndDate()

'version 4.5.2-2025-04-07-2c9fc5bc9, build date 14:59:43 Apr  9 2025'

In [2]:
def picotar(x, J, passo):
    N = x.shape[0]
    Nj = (N-J)//passo + 1
    X = np.zeros((Nj, J))
    for i in range(Nj):
        X[i,:] = x[(i*passo):(i*passo+J)]
    return X

# Lendos os arquivos
Os dados do opensim são da simulação do HamnerTutorial

In [3]:
filepath = 'tibia_r__linear_accelerations.sto'
df = pd.read_csv(filepath, sep='\s+', skiprows=4)
df[['tibia_X', 'tibia_Y', 'tibia_Z']] = df['tibia_r_imu'].str.split(',', expand=True).astype(float)
df = df.drop(columns=['tibia_r_imu'])
df

,time,tibia_X,tibia_Y,tibia_Z
0,0.000,6022.383623,-1030.687761,-17.722485
1,0.017,6410.182684,-445.598930,-305.707043
2,0.033,6130.715217,121.200393,-651.628395
3,0.050,5708.210783,813.416908,-950.006035
4,0.067,5269.245755,1254.659813,-1150.388949
...,...,...,...,...
595,9.917,-2113.201342,1702.562877,-569.900736
596,9.933,-1718.991695,2141.114208,-374.107821
597,9.950,-852.597674,2777.903716,-128.451554
598,9.967,432.726920,3313.785253,26.720697


In [4]:
# Aceleração da tibia direita (X, Y, Z)
fig = px.line(df,
              x='time', 
              y=['tibia_X', 'tibia_Y', 'tibia_Z'], 
              title='Aceleração da Tibia Direita (X, Y, Z)')
fig.show()

In [5]:
filepath = 'Kinematics_BodyKinematics_acc_global.sto'
df_acc_global = pd.read_csv(filepath, sep='\s+', skiprows=18)
df_acc_global

,time,pelvis_X,pelvis_Y,pelvis_Z,pelvis_Ox,pelvis_Oy,pelvis_Oz,femur_r_X,femur_r_Y,femur_r_Z,...,radius_l_Oz,hand_l_X,hand_l_Y,hand_l_Z,hand_l_Ox,hand_l_Oy,hand_l_Oz,center_of_mass_X,center_of_mass_Y,center_of_mass_Z
0,0.000,-311.636450,-1753.645329,-28.681302,-1.149332e+06,1.578603e+06,-5.015982e+06,-684.377416,-1648.439384,862.548676,...,-255085.628715,-410.353078,25.015476,-154.618048,169723.343473,-79054.539848,-255085.628715,0.0,-9.80665,0.0
1,0.017,-443.912581,-1850.404804,6.680045,-1.121261e+06,1.501066e+06,-5.227847e+06,-793.564514,-1945.932582,608.037867,...,-256461.055811,-402.432564,11.222665,-147.082357,164725.278148,-75382.868332,-256461.055811,-0.0,-9.80665,-0.0
2,0.033,-537.960856,-1966.410330,36.880705,-1.073082e+06,1.421026e+06,-5.417507e+06,-1011.912678,-2138.209727,364.818933,...,-261886.404776,-399.165420,-9.272792,-146.396930,165215.584632,-74543.741470,-261886.404776,0.0,-9.80665,0.0
3,0.050,-627.581116,-2064.996565,64.500384,-1.023926e+06,1.322608e+06,-5.547653e+06,-1301.999725,-2156.757398,239.227272,...,-266244.043898,-396.056206,-31.863442,-146.520223,166998.779804,-70740.903717,-266244.043898,0.0,-9.80665,0.0
4,0.067,-662.770613,-2187.219264,80.310277,-1.055866e+06,1.234851e+06,-5.675027e+06,-1600.151124,-2199.821292,176.768388,...,-278429.101592,-407.905197,-52.443368,-157.152968,178787.947688,-69939.793931,-278429.101592,-0.0,-9.80665,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,9.917,-406.830164,-1591.427910,-49.022168,4.617252e+05,1.416464e+05,-4.927566e+06,-3608.013322,118.889225,-550.472325,...,-133797.821366,-123.927127,-261.270929,-191.819261,225272.631705,45957.676639,-133797.821366,-0.0,-9.80665,-0.0
596,9.933,-318.742444,-1473.107112,-44.826179,6.890485e+05,-1.221572e+05,-4.767409e+06,-3558.945542,41.668980,-615.035293,...,-120835.862899,-50.990762,-274.233725,-148.356382,211099.991321,51687.586735,-120835.862899,-0.0,-9.80665,0.0
597,9.950,-247.517596,-1453.976981,-29.341331,9.896820e+05,-4.362015e+05,-4.704232e+06,-3440.228931,-111.462956,-689.945234,...,-119225.432300,10.190755,-296.106714,-109.482145,214121.664699,54128.306543,-119225.432300,-0.0,-9.80665,0.0
598,9.967,-227.210793,-1470.612862,4.878790,1.165629e+06,-7.066692e+05,-4.671996e+06,-3264.189936,-167.612538,-707.228577,...,-115227.728515,56.209675,-304.230370,-76.685811,215058.677416,57042.878392,-115227.728515,0.0,-9.80665,0.0


In [6]:
print(df_acc_global.columns)

Index(['time', 'pelvis_X', 'pelvis_Y', 'pelvis_Z', 'pelvis_Ox', 'pelvis_Oy',
       'pelvis_Oz', 'femur_r_X', 'femur_r_Y', 'femur_r_Z',
       ...
       'radius_l_Oz', 'hand_l_X', 'hand_l_Y', 'hand_l_Z', 'hand_l_Ox',
       'hand_l_Oy', 'hand_l_Oz', 'center_of_mass_X', 'center_of_mass_Y',
       'center_of_mass_Z'],
      dtype='object', length=124)


In [7]:
# Aceleração da Mão Esquerda (X, Y, Z)
fig = px.line(df_acc_global, 
              x='time', 
              y=['hand_l_X', 'hand_l_Y', 'hand_l_Z'], 
              title='Aceleração da Mão Esquerda (X, Y, Z)')

fig.show()

In [8]:
# Aceleração da tibia direita (X, Y, Z)
fig = px.line(df_acc_global,
              x='time', 
              y=['tibia_r_X', 'tibia_r_Y', 'tibia_r_Z'], 
              title='Aceleração da Tibia Direita (X, Y, Z)')
fig.show()

In [9]:
# Aceleração do Torso (X, Y, Z)
fig = px.line(df_acc_global,
                x='time', 
                y=['torso_X', 'torso_Y', 'torso_Z'], 
                title='Aceleração do Torso (X, Y, Z)')
fig.show()

In [10]:
# Aceleração do Pé Esquerdo (X, Y, Z)
fig = px.line(df_acc_global,
              x='time', 
              y=['toes_l_X', 'toes_l_Y', 'toes_l_Z'], 
              title='Aceleração do Pé Esquerdo (X, Y, Z)')
fig.show()

In [11]:
file_path = 'subject02_running.trc'

# 1. Ler apenas os nomes dos marcadores (Linha 4)
# O cabeçalho real de nomes está na 4ª linha (index 3)
marker_header = pd.read_csv(file_path, sep='\t', skiprows=3, nrows=0).columns.tolist()

# 2. Criar nomes de colunas amigáveis (Ex: R.ASIS_X, R.ASIS_Y, R.ASIS_Z)
final_cols = ['Frame', 'Time']
# Os marcadores começam da 3ª coluna no arquivo .trc
markers = [m for m in marker_header if m not in ['Frame#', 'Time'] and not m.startswith('Unnamed')]

for marker in markers:
    final_cols.extend([f"{marker}_X", f"{marker}_Y", f"{marker}_Z"])
final_cols.append('Err')

# 3. Ler os dados (começando da Linha 6)
# Usamos skiprows=5 para pular as 5 linhas de metadados e cabeçalho
df = pd.read_csv(file_path, sep='\t', skiprows=5, names=final_cols[1:])

# Remover colunas extras que podem aparecer por causa de abas sobrando no final do arquivo
df = df.dropna(axis=1, how='all')

print(df.head())

    Time   R.ASIS_X    R.ASIS_Y  R.ASIS_Z   L.ASIS_X    L.ASIS_Y   L.ASIS_Z  \
1  0.000  809.00940  1034.51831 -64.61081  797.31982  1044.15015 -312.26974   
2  0.017  811.67346  1035.32629 -66.21532  797.71240  1042.00757 -314.00708   
3  0.033  813.99536  1035.08667 -67.85238  797.87140  1039.10266 -315.70093   
4  0.050  815.61652  1032.95007 -69.48554  797.62500  1034.84033 -317.28040   
5  0.067  816.15851  1028.41162 -70.97578  796.88501  1028.91284 -318.64362   

   V.Sacral_X  V.Sacral_Y  V.Sacral_Z  ...  L.Toe.Med_Z  R.Temple_X  \
1   613.47882  1070.33386  -198.84880  ...   -200.62943   928.98773   
2   615.48242  1067.93689  -198.11252  ...   -195.70213   928.41205   
3   616.96265  1064.35938  -197.47694  ...   -190.70972   927.82843   
4   617.48016  1058.73071  -197.06030  ...   -185.66602   927.23724   
5   616.73138  1050.69482  -196.98833  ...   -180.71107   926.65271   

   R.Temple_Y  R.Temple_Z  L.Temple_X  L.Temple_Y  L.Temple_Z  Top.Head_X  \
1  1600.05640  -164.1

In [12]:
px.line(df, x='Time', y=['L.Toe.Med_X', 'L.Toe.Med_Y', 'L.Toe.Med_Z'], title='Trajetória do marcador L.Toe.Med (X, Y, Z)').show()

In [13]:
posx, posy, posz = df['L.Toe.Med_X'].values, df['L.Toe.Med_Y'].values, df['L.Toe.Med_Z'].values
t = df['Time'].values
dt = t[1] - t[0]
vx, vy, vz = (posx[2:] - posx[:-2]) / (2 * dt), (posy[2:] - posy[:-2]) / (2 * dt), (posz[2:] - posz[:-2]) / (2 * dt)
ax, ay, az = (posx[2:] - 2*posx[1:-1] + posx[:-2]) / (dt**2), (posy[2:] - 2*posy[1:-1] + posy[:-2]) / (dt**2), (posz[2:] - 2*posz[1:-1] + posz[:-2]) / (dt**2)
# Vamos plotar as acelerações ax, ay e az num mesmo gráfico e comparar com as acelerações do arquivo .sto
# Vamos normalizar as acelerações pela energia para facilitar a comparação visual
ax_norm = ax / np.sqrt(np.sum(ax**2))
ay_norm = ay / np.sqrt(np.sum(ay**2))
az_norm = az / np.sqrt(np.sum(az**2))
ax2_norm = df_acc_global['toes_l_X'] / np.sqrt(np.sum(df_acc_global['toes_l_X']**2))
ay2_norm = df_acc_global['toes_l_Y'] / np.sqrt(np.sum(df_acc_global['toes_l_Y']**2))
az2_norm = df_acc_global['toes_l_Z'] / np.sqrt(np.sum(df_acc_global['toes_l_Z']**2))
fig = go.Figure()
fig.add_trace(go.Scatter(x=t[1:-1], y=ax_norm, mode='lines', name='Aceleração X (numérica)'))
fig.add_trace(go.Scatter(x=t[1:-1], y=ay_norm, mode='lines', name='Aceleração Y (numérica)'))
fig.add_trace(go.Scatter(x=t[1:-1], y=az_norm, mode='lines', name='Aceleração Z (numérica)'))
fig.add_trace(go.Scatter(x=df_acc_global['time'][1:-1], y=ax2_norm[1:-1], mode='lines', name='Aceleração X (STO)'))
fig.add_trace(go.Scatter(x=df_acc_global['time'][1:-1], y=ay2_norm[1:-1], mode='lines', name='Aceleração Y (STO)'))
fig.add_trace(go.Scatter(x=df_acc_global['time'][1:-1], y=az2_norm[1:-1], mode='lines', name='Aceleração Z (STO)'))
fig.update_layout(title='Comparação das Acelerações do marcador L.Toe.Med (numérica vs STO)', xaxis_title='Tempo (s)', yaxis_title='Aceleração Normalizada')
fig.show()

In [14]:
xyz = np.sqrt(ax_norm**2 + ay_norm**2 + az_norm**2)
xyz2 = np.sqrt(ax2_norm**2 + ay2_norm**2 + az2_norm**2)
fig = go.Figure()
fig.add_trace(go.Scatter(x=t[1:-1], y=xyz, mode='lines', name='Aceleração Total (numérica)'))
fig.add_trace(go.Scatter(x=t[1:-1], y=xyz2, mode='lines', name='Aceleração Total (simulada)'))
fig.show()

# Vamos organizar os sinais de aceleração (RW, sto e trc)

In [15]:
# Vamos carregar os dados do RealWorld
with open('C:\\Meu Drive\\Doutorado Unicamp\\Projeto\\github\\preditor-autoencoder\\Udata.pkl', 'rb') as file:
    Udata = pickle.load(file)

# Vamos remover a atividade 'jumping'
actis = ['climbingdown', 'climbingup', 'lying', 'running', 'sitting', 'standing', 'walking']
posis = ['chest', 'forearm', 'head', 'shin', 'thigh', 'upperarm', 'waist']
users = ['proband' + x for x in np.arange(1,16).astype(str)]
# proband2 não tem acc_climbingup_forearm
users.remove('proband2')
# Vamos remover usuários com menos de 21000 amostras
users.remove('proband1')
users.remove('proband4')
users.remove('proband7')
users.remove('proband14')

In [16]:
nu = 3  
nd = 0  # Posição 0, que é o chest
na = 3  # Atividade 3, que é running
eixo = 2
fs = 50
sig = np.array(Udata[nu][nd])[na, :, :3]
x = sig[1000:1150]
px.line(sig)

In [17]:
# Vamos organizar os dados em um DataFrame para facilitar a visualização
# Vamos escolher apenas os acelerômetros de um usuário correndo em todos os domínios (posições dos sensores)
# Vamos escolher apenas 10 segundos de dados (500 amostras, do ponto 3500 até 4000) para facilitar a visualização
nu = 3  
na = 3  # Atividade 3, que é running
df_RW = pd.DataFrame({
    'time': np.arange(500) / fs,
    'chest_X': np.array(Udata[nu][0])[na, 3500:4000, 0],
    'chest_Y': np.array(Udata[nu][0])[na, 3500:4000, 1],
    'chest_Z': np.array(Udata[nu][0])[na, 3500:4000, 2],
    'forearm_X': np.array(Udata[nu][1])[na, 3500:4000, 0],
    'forearm_Y': np.array(Udata[nu][1])[na, 3500:4000, 1],
    'forearm_Z': np.array(Udata[nu][1])[na, 3500:4000, 2],
    'head_X': np.array(Udata[nu][2])[na, 3500:4000, 0],
    'head_Y': np.array(Udata[nu][2])[na, 3500:4000, 1],
    'head_Z': np.array(Udata[nu][2])[na, 3500:4000, 2],
    'shin_X': np.array(Udata[nu][3])[na, 3500:4000, 0],
    'shin_Y': np.array(Udata[nu][3])[na, 3500:4000, 1],
    'shin_Z': np.array(Udata[nu][3])[na, 3500:4000, 2],
    'thigh_X': np.array(Udata[nu][4])[na, 3500:4000, 0],
    'thigh_Y': np.array(Udata[nu][4])[na, 3500:4000, 1],
    'thigh_Z': np.array(Udata[nu][4])[na, 3500:4000, 2],
    'upperarm_X': np.array(Udata[nu][5])[na, 3500:4000, 0],
    'upperarm_Y': np.array(Udata[nu][5])[na, 3500:4000, 1],
    'upperarm_Z': np.array(Udata[nu][5])[na, 3500:4000, 2],
    'waist_X': np.array(Udata[nu][6])[na, 3500:4000, 0],
    'waist_Y': np.array(Udata[nu][6])[na, 3500:4000, 1],
    'waist_Z': np.array(Udata[nu][6])[na, 3500:4000, 2]
})
# Vamos filtrar os sinais com um filtro Butterworth passa-altas com fc = 0.3 Hz
for col in df_RW.columns:
    if col != 'time':
        b, a = signal.butter(5, 0.3, 'hp', fs=fs)
        zi = signal.lfilter_zi(b, a)
        z, _ = signal.lfilter(b, a, df_RW[col], zi=zi*df_RW[col][0])
        df_RW[col] = z
# Vamos normalizar os sinais pela energia dos três eixos combinados
ener = (df_RW.values[:, 1:]**2).sum(axis=0).reshape(-1, 3).sum(axis=1)
for i in range(0, df_RW.shape[1]-1, 3):
    df_RW.iloc[:, i+1:i+4] = df_RW.iloc[:, i+1:i+4] / np.sqrt(ener[i//3])

In [18]:
# Agora vamos organizar os dados do arquivo .sto no mesmo formato do DataFrame do RealWorld
# Mas antes temos de associar os nomes dos sensores do arquivo .sto com as posições dos sensores do RealWorld
# chest -> torso, forearm -> hand_l, head -> torso, shin -> tibia_l, thigh -> femur_l, upperarm -> humerus_l, waist -> pelvis
# Por fim, vamos reamostrar os dados do arquivo .sto para 50 Hz, que é a frequência de amostragem do RealWorld, usando scipy.signal.resample
df_sto = pd.DataFrame({
    'time': np.arange(500) / fs,
    'chest_X': scipy.signal.resample(df_acc_global['torso_X'], 500),
    'chest_Y': scipy.signal.resample(df_acc_global['torso_Y'], 500),
    'chest_Z': scipy.signal.resample(df_acc_global['torso_Z'], 500),
    'forearm_X': scipy.signal.resample(df_acc_global['hand_l_X'], 500),
    'forearm_Y': scipy.signal.resample(df_acc_global['hand_l_Y'], 500),
    'forearm_Z': scipy.signal.resample(df_acc_global['hand_l_Z'], 500),
    'head_X': scipy.signal.resample(df_acc_global['torso_X'], 500),
    'head_Y': scipy.signal.resample(df_acc_global['torso_Y'], 500),
    'head_Z': scipy.signal.resample(df_acc_global['torso_Z'], 500),
    'shin_X': scipy.signal.resample(df_acc_global['tibia_l_X'], 500),
    'shin_Y': scipy.signal.resample(df_acc_global['tibia_l_Y'], 500),
    'shin_Z': scipy.signal.resample(df_acc_global['tibia_l_Z'], 500),
    'thigh_X': scipy.signal.resample(df_acc_global['femur_l_X'], 500),
    'thigh_Y': scipy.signal.resample(df_acc_global['femur_l_Y'], 500),
    'thigh_Z': scipy.signal.resample(df_acc_global['femur_l_Z'], 500),
    'upperarm_X': scipy.signal.resample(df_acc_global['humerus_l_X'], 500),
    'upperarm_Y': scipy.signal.resample(df_acc_global['humerus_l_Y'], 500),
    'upperarm_Z': scipy.signal.resample(df_acc_global['humerus_l_Z'], 500),
    'waist_X': scipy.signal.resample(df_acc_global['pelvis_X'], 500),
    'waist_Y': scipy.signal.resample(df_acc_global['pelvis_Y'], 500),
    'waist_Z': scipy.signal.resample(df_acc_global['pelvis_Z'], 500)
})
# Vamos filtrar os sinais com um filtro Butterworth passa-altas com fc = 0.3 Hz
for col in df_sto.columns:
    if col != 'time':
        b, a = signal.butter(5, 0.3, 'hp', fs=fs)
        zi = signal.lfilter_zi(b, a)
        z, _ = signal.lfilter(b, a, df_sto[col], zi=zi*df_sto[col][0])
        df_sto[col] = z
# Vamos normalizar os sinais pela energia dos três eixos combinados
ener = (df_sto.values[:, 1:]**2).sum(axis=0).reshape(-1, 3).sum(axis=1)
for i in range(0, df_sto.shape[1]-1, 3):
    df_sto.iloc[:, i+1:i+4] = df_sto.iloc[:, i+1:i+4] / np.sqrt(ener[i//3])

In [19]:
# Agora vamos organizar os dados dos marcadores do arquivo .trc no mesmo formato do DataFrame do RealWorld
# Mas antes temos de associar os nomes dos marcadores do arquivo .trc com as posições dos sensores do RealWorld
# chest -> Sternum, forearm -> L.Wrist.Med, head -> L.Acromium, shin -> L.Midfoot.Lat, thigh -> L.Thigh.Upper, upperarm -> L.Bicep, waist -> L.ASIS
# Vamos lembrar de derivar os dados de posição dos marcadores para obter as acelerações e depois reamostrar para 50 Hz
# Vamos criar um mapeamento dos marcadores para as posições dos sensores do RealWorld para todos os eixos (como chest_X -> Sternum_X etc.)
marker_to_sensor = {
    'chest_X': 'Sternum_X',
    'chest_Y': 'Sternum_Y',
    'chest_Z': 'Sternum_Z',
    'forearm_X': 'L.Wrist.Med_X',
    'forearm_Y': 'L.Wrist.Med_Y',
    'forearm_Z': 'L.Wrist.Med_Z',
    'head_X': 'L.Acromium_X',
    'head_Y': 'L.Acromium_Y',
    'head_Z': 'L.Acromium_Z',
    'shin_X': 'L.Midfoot.Lat_X',
    'shin_Y': 'L.Midfoot.Lat_Y',
    'shin_Z': 'L.Midfoot.Lat_Z',
    'thigh_X': 'L.Thigh.Upper_X',
    'thigh_Y': 'L.Thigh.Upper_Y',
    'thigh_Z': 'L.Thigh.Upper_Z',
    'upperarm_X': 'R.Bicep_X',  # O L.Bicep está com todos os valores iguais a zero, então vamos usar o R.Bicep para representar o upperarm
    'upperarm_Y': 'R.Bicep_Y',
    'upperarm_Z': 'R.Bicep_Z',
    'waist_X': 'L.ASIS_X',
    'waist_Y': 'L.ASIS_Y',
    'waist_Z': 'L.ASIS_Z'
}
df_trc = pd.DataFrame({
    'time': np.arange(500) / fs
})
for sensor, marker in marker_to_sensor.items():
    # Derivar os dados de posição para obter a aceleração
    pos = df[marker].values
    vel = np.gradient(pos, dt)
    acc = np.gradient(vel, dt)
    # Reamostrar para 50 Hz
    df_trc[sensor] = scipy.signal.resample(acc, 500)
# Vamos filtrar os sinais com um filtro Butterworth passa-altas com fc = 0.3 Hz
for col in df_trc.columns:
    if col != 'time':
        b, a = signal.butter(5, 0.3, 'hp', fs=fs)
        zi = signal.lfilter_zi(b, a)
        z, _ = signal.lfilter(b, a, df_trc[col], zi=zi*df_trc[col][0])
        df_trc[col] = z
# Vamos normalizar os sinais pela energia dos três eixos combinados
ener = (df_trc.values[:, 1:]**2).sum(axis=0).reshape(-1, 3).sum(axis=1)
for i in range(0, df_trc.shape[1]-1, 3):
    df_trc.iloc[:, i+1:i+4] = df_trc.iloc[:, i+1:i+4] / np.sqrt(ener[i//3])

In [20]:
# Agora vamos visualizar os dados do RealWorld, do arquivo .sto e do arquivo .trc para comparar as acelerações dos sensores
dom = 'thigh'  # Podemos mudar para 'chest', 'forearm', 'head', 'thigh', 'upperarm' ou 'waist' para comparar outros sensores
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_X'], mode='lines', name=f'RealWorld {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Y'], mode='lines', name=f'RealWorld {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Z'], mode='lines', name=f'RealWorld {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_X'], mode='lines', name=f'STO {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Y'], mode='lines', name=f'STO {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Z'], mode='lines', name=f'STO {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_X'], mode='lines', name=f'TRC {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Y'], mode='lines', name=f'TRC {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Z'], mode='lines', name=f'TRC {dom.capitalize()} Z'))
fig.update_layout(title=f'Comparação das Acelerações do Sensor {dom.capitalize()} entre RW, STO e TRC', xaxis_title='Tempo (s)', yaxis_title='Aceleração')
fig.show()

# Vamos analisar o conteúdo espectral dos sinais

In [21]:
# Agora vamos visualizar os dados do RealWorld, do arquivo .sto e do arquivo .trc para comparar as acelerações dos sensores
dom = 'thigh'  # Podemos mudar para 'chest', 'forearm', 'head', 'thigh', 'upperarm' ou 'waist' para comparar outros sensores
J = 150
s = df_trc[f'{dom}_X'].values[:J*2]
cor = np.correlate(s, s, 'full')[-2*J:]
peaks, _ = find_peaks(cor, height=0)
f0 = fs/peaks[cor[peaks].argmax()]
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_X'], mode='lines', name=f'RealWorld {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Y'], mode='lines', name=f'RealWorld {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Z'], mode='lines', name=f'RealWorld {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_X'], mode='lines', name=f'STO {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Y'], mode='lines', name=f'STO {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Z'], mode='lines', name=f'STO {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_X'], mode='lines', name=f'TRC {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Y'], mode='lines', name=f'TRC {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Z'], mode='lines', name=f'TRC {dom.capitalize()} Z'))
fig.update_layout(title=f'Comparação das Acelerações do Sensor {dom.capitalize()} entre RealWorld, STO e TRC', xaxis_title='Tempo (s)', yaxis_title='Aceleração')
fig.update_layout(
    xaxis=dict(
        showgrid=True,
        tickvals= np.arange(1,10*f0)/f0,
        ticktext= (np.arange(1,10*f0)/f0).round(2),
    )
)
fig.show()

In [22]:
# Agora vamos visualizar os dados do RealWorld, do arquivo .sto e do arquivo .trc para comparar as acelerações dos sensores
dom = 'waist'  # Podemos mudar para 'chest', 'forearm', 'head', 'thigh', 'upperarm' ou 'waist' para comparar outros sensores
J = 150
s = df_trc[f'{dom}_X'].values[:J*2]
cor = np.correlate(s, s, 'full')[-2*J:]
peaks, _ = find_peaks(cor, height=0)
f0 = fs/peaks[cor[peaks].argmax()]
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_X'], mode='lines', name=f'RealWorld {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Y'], mode='lines', name=f'RealWorld {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Z'], mode='lines', name=f'RealWorld {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_X'], mode='lines', name=f'STO {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Y'], mode='lines', name=f'STO {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Z'], mode='lines', name=f'STO {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_X'], mode='lines', name=f'TRC {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Y'], mode='lines', name=f'TRC {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Z'], mode='lines', name=f'TRC {dom.capitalize()} Z'))
fig.update_layout(title=f'Comparação das Acelerações do Sensor {dom.capitalize()} entre RealWorld, STO e TRC', xaxis_title='Tempo (s)', yaxis_title='Aceleração')
fig.update_layout(
    xaxis=dict(
        showgrid=True,
        tickvals= np.arange(1,10*f0)/f0,
        ticktext= (np.arange(1,10*f0)/f0).round(2),
    )
)
fig.show()

In [28]:
eixo = 'Y'
s = df_RW[f'{dom}_{eixo}'].values[:J*2]
cor = np.correlate(s, s, 'full')[-2*J:]
px.line(cor).show()

In [23]:
dom = 'waist'
J = 150
dim = 14
vals = np.linspace(0, fs/2, 500)
for eixo in ['X', 'Y', 'Z']:
    s = df_RW[f'{dom}_{eixo}'].values[:J*2]
    X = picotar(s, J, 1)
    Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
    R = Xm.T.dot(Xm)
    l, v = np.linalg.eig(R)
    B = v[:,dim:]
    Pmu = []
    for f in vals:
        e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
        Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
    A = 20*np.log10(np.array(Pmu))
    aux = np.hstack((s, np.zeros(2700)))
    Xf = np.fft.fft(aux)
    freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
    cor = np.correlate(s, s, 'full')[-2*J:]
    peaks, _ = find_peaks(cor, height=0)
    f0 = fs/peaks[cor[peaks].argmax()]
    ind = f0/fs*3000
    inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=freq[:1500],y=20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), mode="lines", showlegend=True, name='fft'))
    fig.add_trace(go.Scatter(x=freq[inds],y=20*np.log10(abs(Xf[inds])/np.sqrt(300)+1e-3), mode="markers", showlegend=True, name='Harmonicos'))
    fig.add_trace(go.Scatter(x=vals,y=A, mode="lines", showlegend=True, name='MUSIC'))
    fig.update_layout(
        xaxis=dict(
            showgrid=True,
            tickvals= np.arange(1,(fs/2)/f0)*f0,
            ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2),
        )
    )
    fig.show()

In [21]:
dom = 'thigh'
J = 150
dim = 14
vals = np.linspace(0, fs/2, 500)
for eixo in ['X', 'Y', 'Z']:
    s = df_trc[f'{dom}_{eixo}'].values[:J*2]
    X = picotar(s, J, 1)
    Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
    R = Xm.T.dot(Xm)
    l, v = np.linalg.eig(R)
    B = v[:,dim:]
    Pmu = []
    for f in vals:
        e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
        Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
    A = 20*np.log10(np.array(Pmu))
    aux = np.hstack((s, np.zeros(2700)))
    Xf = np.fft.fft(aux)
    freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
    cor = np.correlate(s, s, 'full')[-2*J:]
    peaks, _ = find_peaks(cor, height=0)
    f0 = fs/peaks[cor[peaks].argmax()]
    ind = f0/fs*3000
    inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=freq[:1500],y=20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), mode="lines", showlegend=True, name='fft'))
    fig.add_trace(go.Scatter(x=freq[inds],y=20*np.log10(abs(Xf[inds])/np.sqrt(300)+1e-3), mode="markers", showlegend=True, name='Harmonicos'))
    fig.add_trace(go.Scatter(x=vals,y=A, mode="lines", showlegend=True, name='MUSIC'))
    fig.update_layout(
        xaxis=dict(
            showgrid=True,
            tickvals= np.arange(1,(fs/2)/f0)*f0,
            ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2),
        )
    )
    fig.show()

In [44]:
# Agora vamos somar o espectro dos três eixos para tentar melhorar a detecção dos harmônicos
for dom in ['thigh', 'shin', 'upperarm', 'waist', 'chest', 'forearm', 'head']:
    J = 150
    dim = 14
    vals = np.linspace(0, fs/2, 500)
    Spec_final = np.zeros(1500)
    A_spec_final = np.zeros_like(vals)
    for eixo in ['X', 'Y', 'Z']:
        s = df_trc[f'{dom}_{eixo}'].values[:J*2]
        X = picotar(s, J, 1)
        Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
        R = Xm.T.dot(Xm)
        l, v = np.linalg.eig(R)
        B = v[:,dim:]
        Pmu = []
        for f in vals:
            e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
            Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
        A = 20*np.log10(np.array(Pmu))
        aux = np.hstack((s, np.zeros(2700)))
        Xf = np.fft.fft(aux)
        Spec_final += abs(Xf[:1500])
        A_spec_final += np.array(Pmu)
        freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
        cor = np.correlate(s, s, 'full')[-2*J:]
        peaks, _ = find_peaks(cor, height=0)
        f0 = fs/peaks[cor[peaks].argmax()]
        ind = f0/fs*3000
        inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
    A_spec_final = 20*np.log10(A_spec_final)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=freq[:1500],y=20*np.log10(Spec_final/np.sqrt(300)+1e-3), mode="lines", showlegend=True, name='fft soma dos eixos'))
    fig.add_trace(go.Scatter(x=freq[inds],y=20*np.log10(Spec_final[inds]/np.sqrt(300)+1e-3), mode="markers", showlegend=True, name='Harmonicos'))
    fig.add_trace(go.Scatter(x=vals,y=A_spec_final, mode="lines", showlegend=True, name='MUSIC soma dos eixos'))
    fig.update_layout(
        xaxis=dict(
            showgrid=True,
            tickvals= np.hstack([np.arange(0,(fs/2)/f0)*f0, fs/2]),
            ticktext= np.hstack([(np.arange(0,(fs/2)/f0)*f0).round(2), fs/2])
        )
    )
    fig.update_layout(title=f'{dom.capitalize()}')
    fig.update_xaxes(range=[0, 25])
    fig.show()

In [45]:
# Agora vamos somar o espectro dos três eixos para tentar melhorar a detecção dos harmônicos
for dom in ['thigh', 'shin', 'upperarm', 'waist', 'chest', 'forearm', 'head']:
    J = 150
    dim = 14
    vals = np.linspace(0, fs/2, 500)
    Spec_final = np.zeros(1500)
    A_spec_final = np.zeros_like(vals)
    for eixo in ['X', 'Y', 'Z']:
        s = df_sto[f'{dom}_{eixo}'].values[:J*2]
        X = picotar(s, J, 1)
        Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
        R = Xm.T.dot(Xm)
        l, v = np.linalg.eig(R)
        B = v[:,dim:]
        Pmu = []
        for f in vals:
            e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
            Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
        A = 20*np.log10(np.array(Pmu))
        aux = np.hstack((s, np.zeros(2700)))
        Xf = np.fft.fft(aux)
        Spec_final += abs(Xf[:1500])
        A_spec_final += np.array(Pmu)
        freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
        cor = np.correlate(s, s, 'full')[-2*J:]
        peaks, _ = find_peaks(cor, height=0)
        f0 = fs/peaks[cor[peaks].argmax()]
        ind = f0/fs*3000
        inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
    A_spec_final = 20*np.log10(A_spec_final)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=freq[:1500],y=20*np.log10(Spec_final/np.sqrt(300)+1e-3), mode="lines", showlegend=True, name='fft soma dos eixos'))
    fig.add_trace(go.Scatter(x=freq[inds],y=20*np.log10(Spec_final[inds]/np.sqrt(300)+1e-3), mode="markers", showlegend=True, name='Harmonicos'))
    fig.add_trace(go.Scatter(x=vals,y=A_spec_final, mode="lines", showlegend=True, name='MUSIC soma dos eixos'))
    fig.update_layout(
        xaxis=dict(
            showgrid=True,
            tickvals= np.hstack([np.arange(0,(fs/2)/f0)*f0, fs/2]),
            ticktext= np.hstack([(np.arange(0,(fs/2)/f0)*f0).round(2), fs/2])
        )
    )
    fig.update_layout(title=f'{dom.capitalize()}')
    fig.update_xaxes(range=[0, 25])
    fig.show()

In [46]:
# Agora vamos somar o espectro dos três eixos para tentar melhorar a detecção dos harmônicos
for dom in ['thigh', 'shin', 'upperarm', 'waist', 'chest', 'forearm', 'head']:
    J = 150
    dim = 14
    vals = np.linspace(0, fs/2, 500)
    Spec_final = np.zeros(1500)
    A_spec_final = np.zeros_like(vals)
    for eixo in ['X', 'Y', 'Z']:
        s = df_RW[f'{dom}_{eixo}'].values[:J*2]
        X = picotar(s, J, 1)
        Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
        R = Xm.T.dot(Xm)
        l, v = np.linalg.eig(R)
        B = v[:,dim:]
        Pmu = []
        for f in vals:
            e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
            Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
        A = 20*np.log10(np.array(Pmu))
        aux = np.hstack((s, np.zeros(2700)))
        Xf = np.fft.fft(aux)
        Spec_final += abs(Xf[:1500])
        A_spec_final += np.array(Pmu)
        freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
        cor = np.correlate(s, s, 'full')[-2*J:]
        peaks, _ = find_peaks(cor, height=0)
        f0 = fs/peaks[cor[peaks].argmax()]
        ind = f0/fs*3000
        inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
    A_spec_final = 20*np.log10(A_spec_final)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=freq[:1500],y=20*np.log10(Spec_final/np.sqrt(300)+1e-3), mode="lines", showlegend=True, name='fft soma dos eixos'))
    fig.add_trace(go.Scatter(x=freq[inds],y=20*np.log10(Spec_final[inds]/np.sqrt(300)+1e-3), mode="markers", showlegend=True, name='Harmonicos'))
    fig.add_trace(go.Scatter(x=vals,y=A_spec_final, mode="lines", showlegend=True, name='MUSIC soma dos eixos'))
    fig.update_layout(
        xaxis=dict(
            showgrid=True,
            tickvals= np.hstack([np.arange(0,(fs/2)/f0)*f0, fs/2]),
            ticktext= np.hstack([(np.arange(0,(fs/2)/f0)*f0).round(2), fs/2])
        )
    )
    fig.update_layout(title=f'{dom.capitalize()}')
    fig.update_xaxes(range=[0, 25])
    fig.show()

In [38]:
dom = 'waist'
px.line(df_trc, x='time', y=[f'{dom}_{axis}' for axis in ['X', 'Y', 'Z']], title=f'Aceleração do {dom.capitalize()} (X, Y, Z)').show()

# Voltando à estimação de pitch...
Agora é essencial construir a estimativa considerando um sinal multidimensional (triaxial), como no modelo:
$$ \mathbf{x}(t)= \sum_{k=1}^L ​[ a_k \cos(2πkf_0​t) + b_k \sin(2πkf_0​t) ] + ε(t) $$

In [65]:
dom = 'waist'
J = 150
x = df_RW[f'{dom}_X'].values[:J]
y = df_RW[f'{dom}_Y'].values[:J]
z = df_RW[f'{dom}_Z'].values[:J]
m = np.sqrt(x**2 + y**2 + z**2)

In [92]:
dom = 'waist'
J = 150
x = df_trc[f'{dom}_X'].values[:J]
y = df_trc[f'{dom}_Y'].values[:J]
z = df_trc[f'{dom}_Z'].values[:J]
m = np.sqrt(x**2 + y**2 + z**2)

In [33]:
def pcabase(X):
    M = np.ones((X.shape[0],1)).dot(X.mean(axis=0).reshape(1,X.shape[1]))
    R = (X-M).transpose().dot(X-M)
    l, vet = np.linalg.eig(R)
    inds = np.argsort(l)
    vet = vet[:,inds][:,::-1]
    return l, vet

In [93]:
triA = np.array([x, y, z]).T
l, vet = pcabase(triA)
# Vamos projetar os dados nos autovetores para obter as componentes principais
pc1 = triA.dot(vet[:,0])
pc2 = triA.dot(vet[:,1])
pc3 = triA.dot(vet[:,2])
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=x, mode='lines', name='X'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=y, mode='lines', name='Y'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=z, mode='lines', name='Z'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=m, mode='lines', name='Magnitude'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=pc1, mode='lines', name='PC1'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=pc2, mode='lines', name='PC2'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=pc3, mode='lines', name='PC3'))
fig.update_layout(title=f'Componentes Principais do {dom.capitalize()}', xaxis_title='Tempo (s)', yaxis_title='Valor da Componente Principal')
fig.show()
print(f'Variância explicada pela PC1: {l[0]/l.sum()*100:.2f}%')
print(f'Variância explicada pela PC2: {l[1]/l.sum()*100:.2f}%')
print(f'Variância explicada pela PC3: {l[2]/l.sum()*100:.2f}%')

Variância explicada pela PC1: 93.18%
Variância explicada pela PC2: 6.07%
Variância explicada pela PC3: 0.75%


In [69]:
# Agora vamos plotar a nuvem de pontos tridimensional dos dados do sensor para visualizar a forma da trajetória no espaço tridimensional
# Vamos usar cores para representar o tempo na trajetória
# vamos mostrar os autovetores na nuvem de pontos para visualizar as direções das componentes principais
fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(size=5, color=df_RW['time'][:J], colorscale='Viridis', opacity=0.8))])
# Adicionar os autovetores como linhas na nuvem de pontos
fig.add_trace(go.Scatter3d(x=[0, vet[0,0]/20], y=[0, vet[1,0]/20], z=[0, vet[2,0]/20], mode='lines', line=dict(color='red', width=5), name='PC1'))
fig.add_trace(go.Scatter3d(x=[0, vet[0,1]/20], y=[0, vet[1,1]/20], z=[0, vet[2,1]/20], mode='lines', line=dict(color='green', width=5), name='PC2'))
fig.add_trace(go.Scatter3d(x=[0, vet[0,2]/20], y=[0, vet[1,2]/20], z=[0, vet[2,2]/20], mode='lines', line=dict(color='blue', width=5), name='PC3'))
fig.update_layout(title=f'Trajetória Tridimensional do Sensor {dom.capitalize()} com Componentes Principais', scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))
fig.update_layout(width=800, height=600)
# Vamos garantir que os eixos tenham a mesma escala para não distorcer a forma da trajetória
fig.update_layout(scene=dict(aspectmode='data'))
# Vamos adicionar uma barra lateral para mostrar a escala de cores do tempo
fig.update_layout(coloraxis_colorbar=dict(title='Tempo (s)'))
fig.show()

In [101]:
xcor = np.correlate(x, x, 'full')[-J:]
ycor = np.correlate(y, y, 'full')[-J:]
zcor = np.correlate(z, z, 'full')[-J:]
mcor = np.correlate(m - m.mean(), m - m.mean(), 'full')[-J:]
tcor = np.mean([xcor, ycor, zcor], axis=0)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=xcor, mode='lines', name='Autocorrelação X'))
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=ycor, mode='lines', name='Autocorrelação Y'))
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=zcor, mode='lines', name='Autocorrelação Z'))
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=mcor, mode='lines', name='Autocorrelação M'))
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=tcor, mode='lines', name='Autocorrelação média dos eixos'))
fig.update_layout(title=f'Autocorrelação dos Eixos do Sensor {dom.capitalize()}', xaxis_title='Deslocamento (amostras)', yaxis_title='Valor da Autocorrelação')
fig.show()

In [106]:
xcor = np.correlate(pc1, pc1, 'full')[-J:]
ycor = np.correlate(pc2, pc2, 'full')[-J:]
zcor = np.correlate(pc3, pc3, 'full')[-J:]
mcor = np.correlate(m - m.mean(), m - m.mean(), 'full')[-J:]
tcor = np.mean([xcor, ycor, zcor], axis=0)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=xcor, mode='lines', name='Autocorrelação PC1'))
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=ycor, mode='lines', name='Autocorrelação PC2'))
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=zcor, mode='lines', name='Autocorrelação PC3'))
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=mcor, mode='lines', name='Autocorrelação M'))
fig.add_trace(go.Scatter(x=np.arange(J)/fs, y=tcor, mode='lines', name='Autocorrelação média das PCs'))
fig.update_layout(title=f'Autocorrelação dos Eixos do Sensor {dom.capitalize()}', xaxis_title='Deslocamento (amostras)', yaxis_title='Valor da Autocorrelação')
fig.show()

In [110]:
peaks, _ = find_peaks(xcor, height=0)
f0x = fs/peaks[xcor[peaks].argmax()]
peaks, _ = find_peaks(ycor, height=0)
f0y = fs/peaks[ycor[peaks].argmax()]
peaks, _ = find_peaks(zcor, height=0)
f0z = fs/peaks[zcor[peaks].argmax()]
peaks, _ = find_peaks(mcor, height=0)
f0m = fs/peaks[mcor[peaks].argmax()]
f0s = [f0x, f0y, f0z, f0m, 2*f0x, 2*f0y, 2*f0z, 2*f0m, f0x/2, f0y/2, f0z/2, f0m/2]
f0s = list(set(f0s))

In [116]:
L = 6
J = 150
X = np.hstack((x.reshape(-1,1), y.reshape(-1,1), z.reshape(-1,1)))
erro = []
for f0 in f0s:
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(X)
    recon = theta.dot(coefs)
    erro.append(np.sqrt(np.sum((X - recon)**2)))
f0s[np.argmin(erro)]

1.4285714285714286

In [ ]:
L = 6
f0 = 1.43
J = 150
arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
C = np.cos(arg)
S = np.sin(arg)
theta = np.hstack((C, S))
X = np.hstack((x.reshape(-1,1), y.reshape(-1,1), z.reshape(-1,1)))
coefs = np.linalg.pinv(theta).dot(X)
recon = theta.dot(coefs)
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=x, mode='lines', name='Sinal Original x'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=y, mode='lines', name='Sinal Original y'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=z, mode='lines', name='Sinal Original z'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=recon[:, 0], mode='lines', name='Sinal Reconstruído x'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=recon[:, 1], mode='lines', name='Sinal Reconstruído y'))
fig.add_trace(go.Scatter(x=df_trc['time'][:J], y=recon[:, 2], mode='lines', name='Sinal Reconstruído z'))
fig.update_layout(title=f'Reconstrução do Sinal do Sensor {dom.capitalize()} usando Fourier', xaxis_title='Tempo (s)', yaxis_title='Valor')
fig.show()

In [ ]:
# Pergunta de pesquisa: qual informação é compartilhada entre sensores em diferentes posições?
# Aparentemente, a posição (frequência) e a amplitude dos (~seis primeiros) harmônicos parecem ser relevantes.
# Como podemos verificar isso de forma mais quantitativa?
# Podemos tentar codificar os dados usando apenas a posição e amplitude dos harmônicos como features, 
# e comparar a acurácia com um classificador que usa o espectro completo ou os dados brutos. 
# Se a acurácia for similar, isso indicaria que a posição e amplitude dos harmônicos são de fato as informações mais relevantes compartilhadas entre os sensores.
# Nessa abordagem, é crucial ter um método para capturar essas features de forma robusta, 
# especialmente considerando que os harmônicos podem não ser perfeitamente nítidos ou podem variar um pouco entre os sensores.
# Quais métodos de extração de features poderiam ser mais adequados para capturar a posição e amplitude dos harmônicos de forma robusta?
# Poderíamos usar técnicas de detecção de picos no espectro, ou métodos de decomposição como a decomposição em modos empíricos (EMD) para isolar os componentes harmônicos.
# Outra abordagem seria calcular a correlação entre os espectros dos diferentes sensores e 
# verificar se os harmônicos mais fortes são os que apresentam maior correlação entre os sensores.

# Outro ponto essencial, considerando que essas features (posição e amplitude dos harmônicos) sejam realmente relevantes, 
# é considerar como essas features se transformam de um domínio para outro (especificamente considerando um domínio como sendo a posição do sensor)
# Por exemplo, se um sensor no torso tem um harmônico forte em uma frequência específica,
# e um sensor na perna tem um harmônico forte na mesma frequência, isso poderia indicar que ambos os sensores estão capturando a mesma informação de movimento, 
# mesmo que estejam em posições diferentes.
# Isso poderia ser verificado calculando a correlação entre os espectros dos diferentes sensores e verificando se os harmônicos mais fortes são os que 
# apresentam maior correlação entre os sensores.